# Qwen2 — Resmî yazı / karar destek LoRA (Görev 2)

Memurun seçtiği `process_status` ve RAG maddeleriyle resmî yazı taslağı üretir (`draft`, `response_type`, `target_unit`, …).

**Veri:** Drive'daki tam jsonl (`qwen2_train_updated.jsonl` / `qwen2_val_updated.jsonl`). Repoda yalnızca temsili örnek vardır (`yazi_qwen2/veri/*_ornek.jsonl`).

**Yöntem:** Unsloth, native **bfloat16** (4-bit yok), LoRA. Çıktı Drive'daki adaptör klasörü; canlı demoda merge yok (`yazi_lora`).

| Parametre | Değer | Ne işe yarar |
|---|---|---|
| dtype | bfloat16 | Kuantizasyon kaybı yok |
| batch | 32, accum 1 | A100 80 GB |
| MAX_SEQ_LENGTH | 3072 | Uzun taslak + mevzuat |
| Süre | ~15–20 dk | 3 epoch |

**Colab:** Runtime → A100. Kurulum hücresinden sonra **Restart session**, sonra devam.


## 1) Kurulum

Bitince **Runtime → Restart session**. Sonra Drive hücresinden devam edin.


In [ ]:
%%capture
!pip install -q -U unsloth
!pip install -q -U --no-deps trl peft accelerate bitsandbytes
print('Kurulum tamamlandı. Runtime → Restart Runtime yapın!')

## 2) Drive ve veri yolları

`TRAIN_PATH` / `VAL_PATH` kendi Drive klasörünüze göre olsun. Dosya yoksa `assert` durur.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ================================================================
# BURAYA KENDI KLASOR ADINIZI YAZIN:
DATA_DIR = "/content/drive/MyDrive/qwen2_egitim_seti"
# ================================================================

TRAIN_PATH  = os.path.join(DATA_DIR, "qwen2_train_updated.jsonl")
VAL_PATH    = os.path.join(DATA_DIR, "qwen2_val_updated.jsonl")
TEST_PATH   = os.path.join(DATA_DIR, "qwen2_test_updated.jsonl")
OUTPUT_DIR  = os.path.join(DATA_DIR, "qwen25_7b_finetuned")

for label, path in [("TRAIN", TRAIN_PATH), ("VAL", VAL_PATH), ("TEST", TEST_PATH)]:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {label}: {path}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nOUTPUT_DIR: {OUTPUT_DIR}")

## 3) GPU kontrolü ve taban modeli yükle

16-bit BF16. A100 bellek bütçesi bu ayara göre.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 3072
DTYPE          = torch.bfloat16
LOAD_IN_4BIT   = False  # A100 80GB → native 16-bit

print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16  : {torch.cuda.is_bf16_supported()}")
print("Model yükleniyor...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype        = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

print("\n✅ Model yüklendi!")

## 4) LoRA katmanlarını ekle

Taban donuk kalır; yalnız adaptörler eğitilir.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 64,
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)

print("✅ LoRA adaptörleri eklendi.")
model.print_trainable_parameters()

## 5) Veriyi ChatML metnine çevir

jsonl `messages` listesi tokenizer şablonuna dökülür.


In [ ]:
from datasets import load_dataset
import multiprocessing

train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_ds   = load_dataset("json", data_files=VAL_PATH,   split="train")

def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

num_cpus = multiprocessing.cpu_count()
print(f"CPU çekirdek sayısı: {num_cpus}")

train_ds = train_ds.map(formatting_func, num_proc=num_cpus)
val_ds   = val_ds.map(formatting_func,   num_proc=num_cpus)

print(f"\n✅ Train: {len(train_ds)} kayıt, Val: {len(val_ds)} kayıt")
print(train_ds[0]["text"][:400])

## 6) SFT ayarları

- batch 32, accum 1 → efektif 32
- `bf16=True`
- `group_by_length` padding'i kısar
- loss yanıt (taslak JSON) üzerinde


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    # --- A100 80GB MAX HIZ ---
    per_device_train_batch_size  = 32,  # 80GB'da rahat → grad_accum=1 ile tek adımda update
    per_device_eval_batch_size   = 32,
    gradient_accumulation_steps  = 1,   # Efektif batch=32, tek geçişte güncelleme = en hızlı
    # --- Öğrenme ---
    warmup_ratio                 = 0.05,
    num_train_epochs             = 3,
    learning_rate                = 1e-4,
    lr_scheduler_type            = "cosine",
    weight_decay                 = 0.01,
    # --- Hassasiyet ve Hız ---
    bf16                         = True,   # A100 Native BF16 Tensor Core
    fp16                         = False,
    # --- Logging ve Kayıt ---
    logging_steps                = 5,
    optim                        = "adamw_8bit",
    seed                         = 3407,
    output_dir                   = "outputs",
    save_strategy                = "epoch",
    eval_strategy                = "epoch",
    # --- Veri Yükleme (High-RAM max verim) ---
    group_by_length              = True,   # padding tasarrufu ~%30 hız artışı
    dataloader_num_workers       = 8,      # High-RAM: 8 CPU worker paralel veri
    dataloader_pin_memory        = True,   # RAM→GPU transfer hızlandırma
    report_to                    = "none",
    dataset_text_field           = "text",
    max_seq_length               = MAX_SEQ_LENGTH,
    packing                      = False,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args          = training_args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Adım başına VRAM kullanımını göster
steps_per_epoch = len(train_ds) // 32
print(f"✅ Trainer hazır.")
print(f"   Epoch başına adım : {steps_per_epoch}")
print(f"   Toplam adım       : {steps_per_epoch * 3}")
print(f"   Efektif batch     : 32")

## 7) Eğitimi başlat

A100 80 GB, 3 epoch, yaklaşık 15–20 dakika.


In [ ]:
# Eğitim öncesi GPU durumunu yazdır
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Eğitim öncesi VRAM: {allocated:.1f} GB kullanımda / {total:.1f} GB toplam")
print(f"Eğitim başlıyor...\n")

trainer_stats = trainer.train()

sure_dk = trainer_stats.metrics['train_runtime'] / 60
print(f"\n✅ Eğitim tamamlandı!")
print(f"   Toplam süre  : {sure_dk:.1f} dakika")
print(f"   Train loss   : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   Samples/sn   : {trainer_stats.metrics.get('train_samples_per_second', 'N/A')}")

## 8) Adaptörü Drive'a kaydet

Demo bu klasörü `yazi_lora` olarak yükler. Merge etmeyin.


In [ ]:
# LoRA adaptörü (küçük, ~250 MB)
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"✅ LoRA adaptörü: {OUTPUT_DIR}/lora_adapter")

# Birleştirilmiş 16-bit model (02_evaluate için)
model.save_pretrained_merged(
    f"{OUTPUT_DIR}/merged_16bit",
    tokenizer,
    save_method = "merged_16bit"
)
print(f"✅ Merged 16-bit: {OUTPUT_DIR}/merged_16bit")
print("\nSonraki adım: 02_evaluate_qwen25_7b.ipynb")